# Database Management Systems: Week 9 - In-Depth Notes

## Week 9 Overview: Indexing and Hashing for Efficient Data Access

Week 9 focuses on **indexing** and **hashing**—the two primary techniques for making database search, insert, and delete operations efficient. Building on the data structures and physical storage concepts from Week 8, this week covers:

1. **Indexing Fundamentals** (Module 41): Ordered indices, primary/secondary, dense/sparse, multi-level.
2. **Balanced Trees and 2-3-4 Trees** (Module 42): Precursor to B-trees, guaranteeing O(log n) height.
3. **B+ Tree Index Files** (Module 43): The standard indexing structure for databases.
4. **Hashing** (Module 44): Static and dynamic hashing, bitmap indices.
5. **Index Design** (Module 45): SQL syntax for creating indexes and practical guidelines.

The core goal is to understand how a DBMS can locate any record quickly—typically in O(log n) time—rather than scanning entire tables linearly.

---

## Module 41: Indexing and Hashing – Part 1: Indexing Fundamentals

### 41.1. The Need for Indexing

Consider a simple table with two fields: **name** and **phone number**.

| Record Number | Name | Phone Number |
|---|---|---|
| 1 | Prabir Kumar Biswas | 84772 |
| 2 | Pabitra Mitra | 84773 |
| 3 | Probitra Mitra | 84774 |
| ... | ... | ... |

**Problem:** We frequently need to search by name or by phone number. Neither field is sorted. So, a linear search is O(n), which is unacceptable for large tables.

**Naive solutions:**
- Sort by name: Makes name search O(log n), but phone number search remains O(n).
- Sort by phone: Makes phone search O(log n), but name search becomes O(n).

We cannot sort the same file by both attributes simultaneously.

**Indexing Solution:** Create **secondary tables (index files)** that map search keys to record pointers. The original records remain in their natural order; the index provides sorted access.

### 41.2. Basic Concept of Indexing

An **index** is a separate data structure that associates a **search key** (one or more attributes) with a **pointer** to the corresponding record.

**Example: Index on Name**

Create an index file containing:
```
(Name, Record Number)
(Pabitra Mitra, 2)
(Prabir Kumar Biswas, 1)
(Probitra Mitra, 3)
```

This index is **sorted by name**. To search for "Probitra Mitra":
1. Do a binary search on the index (O(log n)).
2. Find the record number (3).
3. Directly access record 3 in the original file (O(1)).

**Example: Index on Phone Number**

Create another index file sorted by phone number:
```
(Phone Number, Record Number)
(84772, 1)
(84773, 2)
(84774, 3)
```

Now, searching by phone number is also O(log n).

**Key insights:**
1. The original file's physical order is irrelevant.
2. We can create an index on **any attribute** to speed up queries involving that attribute.
3. There is storage overhead and update overhead (index must be maintained).

### 41.3. Types of Indices

There are two broad categories:

1. **Ordered Indices**: Index entries are sorted by the search key.
2. **Hash Indices**: Uses a hash function to map search keys to buckets.

This module focuses on **ordered indices**.

### 41.4. Primary vs Secondary Indices

**Primary Index (Clustering Index):**
- In a **sequentially ordered file**, the index whose search key specifies the sequential order.
- The records themselves are sorted by this key.
- Usually the primary key, but not necessarily.

**Secondary Index (Non-clustering Index):**
- An index whose search key is different from the sequential order of the file.
- Points to records that are stored in a different order.

**Index Sequential File:**
- An ordered sequential file with a primary index.

### 41.5. Dense vs Sparse Indices

#### 41.5.1. Dense Index

- Contains an index entry for **every search key value**.
- One-to-one mapping between index entries and records.

**Example (Index on ID):**
```
Index:                 Data File:
(10101, ptr1)  →       Record 10101
(10200, ptr2)  →       Record 10200
(10300, ptr3)  →       Record 10300
```

**Characteristics:**
- Fast lookup: direct pointer to record.
- Large index size (as large as the data file).
- More maintenance on insert/delete.

**Dense index on non-key attributes:**
If the search key is not unique (e.g., department name), multiple records may have the same key value. A dense index may have multiple entries with the same key.

#### 41.5.2. Sparse Index

- Contains index entries for **only some search key values** (e.g., every 5th record, or the first record of each block).

**Example:**
```
Index:                 Data File:
(10101, block1)  →     Block 1: 10101, 10200, 10300
(15151, block2)  →     Block 2: 15151, 16162, 17293
(21312, block3)  →     Block 3: 21312, 22222, 23333
```

**Search with sparse index:**
1. Binary search the index to find the largest key ≤ search key.
2. Search linearly from that position in the data file.

**Characteristics:**
- Smaller index (1/5th or 1/100th the size of a dense index).
- More search time (must do linear search within a range).
- Less maintenance.

**Important:** Sparse indices require the data file to be **sorted** by the index key. Dense indices can work with any order.

### 41.6. Multi-Level Indices

**Problem:** What if the index itself is too large to fit in memory?

**Solution:** Create an index on the index—a **multi-level index**.

- **Inner Index**: The primary index (e.g., a dense index on the data).
- **Outer Index**: A sparse index on the inner index.

**Structure:**
```
Outer Index (sparse)
    ↓
Inner Index (dense)
    ↓
Data File
```

**Why sparse?** The outer index only needs to point to blocks of the inner index, not every entry.

**Extension:** If the outer index is still too large, create another level. This leads to a **B+ tree** (discussed later).

### 41.7. Secondary Index with Duplicates

When indexing on a non-key attribute (e.g., salary), multiple records may have the same value.

**Problem:** The index entry for a value must point to multiple records.

**Solution:** Use **indirection**:
- Index entries point to a **bucket** containing pointers to all records with that value.
- Or, the index can have multiple identical entries.

**Example (Index on Salary):**
```
Index Entry: (80000) → List of pointers → Record 1 (80000)
                                        → Record 2 (80000)
```

**Important:** Secondary indices must be **dense** (every search key value appears in the index) because the data file is not sorted by the secondary key.

### 41.8. Update Operations on Indices

Every insert, delete, or update of a record requires updating **all** indices on that relation.

**Insertion:**
- Perform a lookup using the search key.
- For dense index: Insert the search key if not present.
- For sparse index: Insert only if a new block is created.

**Deletion:**
- For dense index: Delete the search key if it was the only occurrence.
- For sparse index: If the deleted record's key was in the index, replace it with the next record's key.

**Overhead:** Updating indices imposes overhead. This is the trade-off for faster searches.

### 41.9. Index Evaluation Metrics

When deciding on indices, evaluate:

1. **Access Types Supported Efficiently:**
   - Point queries (specific value)
   - Range queries (between two values)

2. **Access Time:** Time to find a record.

3. **Insertion/Deletion Time:** Time to update the index.

4. **Space Overhead:** Additional storage for the index.

5. **Overflow Handling:** How does the index handle duplicates and growth?

---

## Module 42: Indexing and Hashing – Part 2: Balanced Trees and 2-3-4 Trees

### 42.1. Recap: Binary Search Trees

We know that a Binary Search Tree (BST) has:
- Search: O(h) where h is the height.
- Insert/Delete: O(h) (search + O(1) pointer manipulation).

**The critical problem:** BST height can be O(n) in the worst case (spine tree), making search O(n). We need **balanced** trees where height is O(log n).

### 42.2. Ensuring Balance

Various strategies ensure O(log n) height:

**Worst-Case Guarantee:**
- **AVL Trees**: Self-balancing BST where the height difference between left and right subtrees at any node is at most 1. Requires rotations to rebalance.
- **Red-Black Trees**: Another self-balancing BST with guaranteed O(log n) height.

**Randomized Guarantee:**
- **Randomized BSTs**: Randomize insertion order to achieve O(log n) expected height.
- **Skip Lists**: Multiple levels of linked lists; randomization chooses which levels an element appears in.

**Amortized Guarantee:**
- **Splay Trees**: No strict balancing; guarantees O(log n) amortized cost over a sequence of operations.

**Limitations for Databases:**
- These are **in-memory** data structures.
- They work for 10,000–100,000 records, not millions.
- Complex rotations are difficult to scale to disk.
- They do not map well to external storage (disk-based) structures.

### 42.3. 2-3-4 Trees: A Foundation for B-Trees

The **2-3-4 tree** introduces a key idea that extends naturally to disk-based structures: **all leaves are at the same depth**.

**Node Types:**
1. **2-node**: 1 data value, 2 children.
   - Values in left child < data < values in right child.

2. **3-node**: 2 data values (S and L, S < L), 3 children.
   - Left child: values < S
   - Middle child: S < values < L
   - Right child: values > L

3. **4-node**: 3 data values (S, M, L), 4 children.
   - Child 1: values < S
   - Child 2: S < values < M
   - Child 3: M < values < L
   - Child 4: values > L

**Property:** All leaves are at the same level (height). This guarantees height = O(log n).

### 42.4. Search in a 2-3-4 Tree

Search is a natural extension of BST search:
1. Start at the root.
2. Compare with the data values in the node.
3. Determine which child to descend to.
4. Repeat until found or leaf reached.

**Complexity:** O(h) = O(log n) because all leaves are at the same level.

### 42.5. Insertion in a 2-3-4 Tree

**Algorithm:**
1. Search for the insertion position (where the key would be).
2. If the node is a **2-node**: Change it to a 3-node by adding the new value.
3. If the node is a **3-node**: Change it to a 4-node by adding the new value.
4. If the node is a **4-node**: 
   - **Split** the 4-node: take the middle value M, create two 2-nodes (S and L), and **promote M to the parent**.
   - If no parent (root): Create a new root containing M.
   - Then insert the new value.

**Splitting a 4-node at the root:**
```
Before:           After:
     [S|M|L]          [M]
     / | \ \          /   \
   [a] [b] [c] [d]  [S]   [L]
                   / \   / \
                 [a] [b] [c] [d]
```

**Splitting a 4-node with a 2-node parent:**
```
Before:           After:
    [P]              [P|M]
    / \              / | \
  [S|M|L]      [S]   [L]   [e]
  / | \ \      / \   / \
 [a][b][c][d]  [a][b][c][d]
```

**Height increase:** The **only** time the height of a 2-3-4 tree increases is when the root is split. Since splitting the root increases the level of all leaves uniformly, the property "all leaves at the same level" is preserved.

**Why this matters:** The height only increases when the root is split, which happens after many insertions (when the root becomes a 4-node). This is fundamentally different from ordinary BSTs where height can increase with every insertion.

### 42.6. Insertion Strategies

**Early Splitting:** Split 4-nodes during the descent, before reaching the insertion position. Ensures no 4-node is encountered during recursion.

**Late Splitting:** Split a 4-node only when you need to insert into it.

Both strategies produce valid 2-3-4 trees with O(log n) height, but they may result in different tree structures.

### 42.7. Deletion in 2-3-4 Trees

Deletion is the reverse of insertion:
- Replace 4-node splits with **merges**.
- If a node becomes too empty (1-node), merge with a sibling or borrow from a sibling.
- The height can decrease only when the root merges.

### 42.8. Generalizing to Larger Nodes

The 2-3-4 tree uses three types of nodes (2, 3, 4). This is cumbersome because we must handle multiple node types.

**Alternative:** Use a single type of node that can hold **up to n values and n+1 children**, but require at least **n/2 values**. This is the fundamental idea behind **B-trees and B+ trees**.

**Advantages of 2-3-4 trees:**
- All leaves at same level → O(log n) height guaranteed.
- Data kept in sorted order.
- Generalizes to larger nodes.

**Disadvantages:**
- Multiple node types cause overhead.
- Frequent destruction and construction of nodes.

---

## Module 43: Indexing and Hashing – Part 3: B+ Tree Index Files

### 43.1. Introduction to B+ Trees

A **B+ tree** is a balanced tree structure that extends the 2-3-4 tree idea to disk-based storage.

**Key properties:**
- All leaf nodes are at the same height.
- Internal nodes contain keys and pointers to children.
- Leaf nodes contain keys and pointers to actual data records.
- Leaf nodes are linked together in sorted order (for sequential access).
- Each node can have multiple keys (not just 1-3 like 2-3-4 trees).

### 43.2. B+ Tree Structure

**Internal (non-leaf) Nodes:**
- Contain up to n keys and n+1 pointers.
- Keys are sorted: K1 < K2 < ... < Kn.
- Pointer Pi points to a subtree where all keys are < Ki.
- Pointer Pn+1 points to a subtree where all keys ≥ Kn.

**Leaf Nodes:**
- Contain up to n keys and n pointers to records.
- Each key Ki has a pointer Pi to the corresponding record.
- An extra pointer points to the next leaf node (for sequential access).

**Occupancy Requirements (for n maximum keys):**
- Root: At least 2 children (or 1 if the tree has only one node).
- Internal nodes: At least ⌈n/2⌉ keys (and ⌈n/2⌉ + 1 children).
- Leaf nodes: At least ⌈n/2⌉ keys.

This ensures that nodes are at least **half full**, preventing excessive height and space waste.

### 43.3. Search in a B+ Tree

**Algorithm:**
1. Start at the root.
2. At an internal node: Find the smallest i such that K_i > search_key (or i = n+1 if all keys are less). Follow pointer P_i.
3. At a leaf node: Search linearly among the keys. If found, return the pointer. If not, search fails.

**Complexity:** O(log_{⌈n/2⌉}(K)) where K is the number of keys. Since n is typically large (e.g., 50-200), the height is small.

**Example:** For 1 million keys and n=100, height ≈ log₅₀(1,000,000) ≈ 4. A binary tree would require ≈ 20 levels.

### 43.4. Insertion in a B+ Tree

**Algorithm:**
1. Search for the leaf node where the key should be inserted.
2. If the leaf node has space: Insert the key.
3. If the leaf node is full:
   - **Split** the node into two: one with ⌈n/2⌉ keys, one with ⌈n/2⌉ + 1 keys (or as close as possible).
   - Promote the smallest key of the right node to the parent.
   - If the parent is full, split it recursively.
   - If the root is split, create a new root.

**Guarantee:** After insertion, all nodes maintain at least ⌈n/2⌉ occupancy (except root). Height only increases when the root is split.

### 43.5. Deletion in a B+ Tree

**Algorithm:**
1. Search for the key in the leaf node.
2. Remove the key and its pointer.
3. If the leaf node becomes underfull (< ⌈n/2⌉):
   - Try to **borrow** a key from an adjacent sibling.
   - If borrowing is not possible, **merge** with a sibling and update the parent.
4. If the parent becomes underfull, propagate the merge upward.
5. If the root becomes empty (or has only one pointer), reduce height.

**Guarantee:** All nodes remain at least half full. Height only decreases when the root merges.

### 43.6. B+ Tree as Index File vs Data File

**Index File:**
- Used for secondary indices.
- Leaf nodes contain pointers to records in a separate data file.
- The data file may be a heap, sequential, or another structure.

**Data File (B+ tree file organization):**
- The B+ tree itself stores the actual records.
- Leaf nodes contain the full records (or at least the indexed fields plus the record).
- The B+ tree serves as the primary storage structure.
- This is called **B+ tree file organization**.

**Advantages:**
- No need for a separate index and data file.
- Records are automatically sorted by the primary key.
- No overflow blocks or reorganization needed.

### 43.7. Duplicate Search Keys

If the search key is not unique, multiple records may have the same key value.

**Approaches:**
1. **Allow duplicate keys** in the B+ tree. The leaf nodes may have multiple entries with the same key.
2. **Use a bucket per key**: Each key has a list of record pointers.
3. **Make the key unique**: Add the record's primary key (or a unique record ID) to the search key to ensure uniqueness. This is simpler for insertion/deletion but adds storage overhead.

### 43.8. Record Relocation and Secondary Indices

If a record's location changes (e.g., due to reorganization of the primary file), all secondary index pointers to that record must be updated.

**Better approach:** Use the **primary index search key** as the target of secondary indices, not the physical record pointer. The secondary index points to the primary key value; then a search on the primary index finds the record. This adds an extra lookup but avoids updating secondary indices when records move.

### 43.9. String Indexing and Prefixing

Strings can be variable length, complicating indexing.

**Prefixing technique:** Store only a prefix of the string (e.g., "Silb" for "Silberschatz") in the internal nodes. This reduces space and allows more keys per node (higher fanout).

### 43.10. B-Trees vs B+ Trees

**B-Tree:**
- Search keys appear **only once** in the entire tree.
- Internal nodes may contain record pointers directly.

**B+ Tree:**
- Search keys appear **multiple times** (once in each internal node on the path, and once in the leaf).
- Only leaf nodes contain record pointers.

**Comparison:**

| Aspect | B+ Tree | B-Tree |
|---|---|---|
| Space | More (keys repeated) | Less (keys appear once) |
| Record lookup | Always must go to leaf | May find at internal node |
| Fanout | Higher (internal nodes only store keys) | Lower (internal nodes store keys + pointers) |
| Insert/Delete | Simpler | More complex |
| Sequential access | Easy (linked leaves) | Not directly |
| **Popularity** | **Much more popular** | Less popular |

**Why B+ trees dominate:** The advantages of B-trees (less space, sometimes faster lookup) are outweighed by the advantages of B+ trees (higher fanout → shorter height, simpler algorithms, linked leaves for sequential access).

### 43.11. Typical B+ Tree Parameters

- **Block size**: 4KB to 32KB.
- **n (maximum keys per node)**: 50 to 500, depending on key size and block size.
- **Height**: 2-4 for most databases.
- **Fanout**: At least ⌈n/2⌉.

For n=100 and K=1,000,000 records, height ≈ 4. For K=1,000,000,000, height ≈ 5. This is the power of B+ trees.

---

## Module 44: Indexing and Hashing – Part 4: Hashing

### 44.1. The Idea of Hashing

**Problem:** We want O(1) access to records based on a key value. Array indexing gives O(1) but the domain of key values is often too large (e.g., names, ID numbers).

**Solution:** Use a **hash function** that maps the large domain D to a small range of **bucket indices** (0 to n).

**Hash Function:** h: D → {0, 1, 2, ..., n}

**Example:**
```
Names: John Smith, Lisa Jones, Sam Williams, Sandra Dee
Hash values: h(John Smith) = 2, h(Lisa Jones) = 1, h(Sam Williams) = 4, h(Sandra Dee) = 2
```

**Collision:** When two distinct keys hash to the same value (e.g., John Smith and Sandra Dee both hash to 2).

### 44.2. Buckets

A **bucket** is a storage unit (typically a disk block) that can hold one or more records. The hash function maps a key to a bucket.

**Search:**
1. Compute hash(key) to find the bucket.
2. Search linearly within the bucket.

**Collision handling:** Multiple keys may map to the same bucket. The bucket is searched sequentially.

### 44.3. Hash Function Example

**Hash on Department Name:**

Suppose we have 10 buckets (0 to 9). Hash function: sum of ASCII values of characters in department name, modulo 10.

| Department | ASCII Sum | Hash Value |
|---|---|---|
| Music | ... | 1 |
| History | ... | 2 |
| Computer Science | ... | 6 |
| Physics | ... | 3 |
| Electrical Engineering | ... | 3 |

Physics and Electrical Engineering collide (both hash to 3).

### 44.4. Properties of Good Hash Functions

1. **Uniform Distribution**: Keys should be evenly spread across buckets.
2. **Fast Computation**: The hash function should be quick to compute.
3. **Deterministic**: Same key always produces the same hash value.

**Ideal hash function**: Random function that assigns each key to a uniformly random bucket. In practice, we use pseudo-random or well-designed functions.

### 44.5. Bucket Overflow

**Overflow occurs when:**
- Insufficient number of buckets (range too small)
- Skewed data distribution
- Multiple records with the same key value
- Poor hash function

**Handling overflow:**
- **Overflow buckets (closed hashing)**: When a bucket is full, create a new block and link it. This is like a chain.
- Disadvantage: More block transfers (may need to traverse the chain).

### 44.6. Static Hashing: Problems

**Static hashing** uses a fixed set of buckets (fixed range of hash values).

**Problem with growth:**
- If the database grows, buckets become full, overflow chains become long.
- If we initially allocate many buckets, space is wasted when the database is small.
- To change the number of buckets, we must change the hash function (e.g., change the modulo divisor) and **rehash everything**.

**Conclusion:** Static hashing works well only if the data size is relatively constant.

### 44.7. Dynamic (Extendable) Hashing

**Extendable hashing** overcomes the limitations of static hashing by allowing the number of buckets to grow and shrink dynamically.

**Key ideas:**
1. **Hash function generates a large range**: h(key) produces a b-bit integer (e.g., 32 bits). This is the **virtual** hash space.

2. **Use only a prefix**: We use only the first **i bits** of the hash value to determine the bucket. The rest are ignored.

3. **Bucket Address Table**: An array of pointers indexed by the i-bit prefix. It maps prefixes to actual buckets.

4. **Dynamic growth**: When a bucket overflows:
   - **Increase i by 1** (if the table is too small to distinguish buckets).
   - **Split the overflowing bucket**: Reorganize records between the old bucket and a new one.
   - Double the bucket address table (if i increases).

**Example:**
- b = 32 (hash is a 32-bit integer)
- Initially i = 1: 2 buckets (0 and 1)
  - Prefix 0: buckets 0, 2, 4, ... (all with first bit 0)
  - Prefix 1: buckets 1, 3, 5, ... (all with first bit 1)
- If bucket 1 overflows:
  - Increase i to 2: 4 buckets (00, 01, 10, 11)
  - Split bucket 1 into 10 and 11
  - Reorganize: records with prefix 10 go to one bucket, 11 to another

**Key points:**
- The hash function never changes; only the number of prefix bits used changes.
- Splitting is **local** (only the overflowing bucket is split).
- The bucket address table doubles in size, but the number of actual buckets increases by 1.

### 44.8. Extendable Hashing: Detailed Example

**Initial state (i = 0):** 1 bucket, all keys map to it.

Insert records:
- Mozart (Music): hash = 0... (first bit 0)
- Srinivasan (Computer Science): hash = 1... (first bit 1)
- Wu (Finance): hash = 1... (first bit 1)

After inserting, i = 1, 2 buckets:
- Bucket 0: Mozart
- Bucket 1: Srinivasan, Wu

**Insert Einstein (Physics):** hash = 1...
- Bucket 1 is full (size 2).
- Increase i to 2 (4 buckets: 00, 01, 10, 11).
- Reorganize:
  - Music: 0... → bucket 00
  - Computer Science: 1 1... → bucket 11
  - Finance: 1 0... → bucket 10
  - Physics: 1 0... → bucket 10
- Result:
  - Bucket 00: Mozart (prefix 0)
  - Bucket 11: Srinivasan (prefix 1 1)
  - Bucket 10: Wu, Einstein (prefix 1 0)

The bucket address table now has 4 entries, but only 3 actual buckets. Prefix 01 maps to the same bucket as 00 (since no key starts with 01 yet).

### 44.9. Deletion in Extendable Hashing

- Search for the key and remove it.
- If the bucket becomes empty, merge it with a sibling if possible.
- If merging reduces the need for buckets, decrease i (shrink the bucket address table).

**Trade-off:** Merging is not always done immediately; sometimes it is deferred to reduce overhead.

### 44.10. Comparison: Ordered Indexing vs Hashing

| Aspect | Ordered Indexing (B+ tree) | Hashing |
|---|---|---|
| Point query (specific value) | O(log n) | O(1) average |
| Range query | Efficient (sorted) | Inefficient (scattered) |
| Sequential access | Efficient (linked leaves) | Not supported |
| Insert/Delete | O(log n) | O(1) average |
| Periodic reorganization | Not required (self-balancing) | Not required (extendable) |
| Space overhead | Moderate | Moderate |

**When to use:**
- **Ordered Indexing**: For range queries and sequential access (e.g., find all employees with salary between 50k and 60k).
- **Hashing**: For point queries only (e.g., find the record with ID = 12345).

### 44.11. Bitmap Indices

**Motivation:** For attributes with a small, fixed set of values (e.g., gender, semester, country), a bitmap index can be very efficient.

**Concept:**
- For each distinct value of the attribute, create a **bitmap**—an array of bits, one per record.
- Bit = 1 if the record has that value; 0 otherwise.

**Example: Gender attribute**
```
Record:  1   2   3   4   5
Gender:  M   F   F   M   F

Bitmap for M: 1 0 0 1 0
Bitmap for F: 0 1 1 0 1
```

**Queries:**
- Find all males: Use bitmap for M.
- Find all females: Use bitmap for F.
- Find all males with income level L1: Bitmap for M AND Bitmap for L1.

**Advantages:**
- Fast for equality and logical operations (AND, OR, NOT).
- Compact when the attribute has few distinct values.

**Disadvantages:**
- Not suitable for attributes with many distinct values.
- Updates are more expensive (must update multiple bitmaps).

**Creating a bitmap index in SQL:**
```sql
CREATE BITMAP INDEX emp_gender_idx ON employee (gender);
```

---

## Module 45: Indexing and Hashing – Part 5: Index Design

### 45.1. Creating Indexes in SQL

**Basic Syntax:**
```sql
CREATE INDEX index_name ON table_name (column1, column2, ...);
```

**Example:**
```sql
CREATE INDEX b_index ON branch (branch_name);
```

**UNIQUE Index:**
```sql
CREATE UNIQUE INDEX emp_id_idx ON employee (emp_id);
```
This enforces uniqueness on the indexed column.

**Dropping an Index:**
```sql
DROP INDEX index_name;
```

### 45.2. Index Options

**TABLESPACE:**
```sql
CREATE INDEX emp_name_idx ON employee (ename) TABLESPACE index_tbs;
```
Specifies the tablespace where the index is stored.

**STORAGE options:**
```sql
CREATE INDEX emp_name_idx ON employee (ename)
STORAGE (INITIAL 20K NEXT 20K PCTINCREASE 75);
```
- `INITIAL 20K`: First extent is 20KB.
- `NEXT 20K`: Second extent is 20KB.
- `PCTINCREASE 75`: Subsequent extents grow by 75% of the previous extent.

**PCTFREE:**
```sql
CREATE INDEX emp_name_idx ON employee (ename) PCTFREE 0;
```
Specifies the percentage of each block to leave free for future updates.

**COMPUTE STATISTICS:**
```sql
CREATE INDEX emp_name_idx ON employee (ename) COMPUTE STATISTICS;
```
Collects statistics about the index for the optimizer.

### 45.3. Composite Indices

**Creating an index on multiple columns:**
```sql
CREATE INDEX emp_dept_sal_idx ON employee (dept_name, salary);
```

**Order matters:**
- The leftmost column is the primary ordering key.
- `(dept_name, salary)` is different from `(salary, dept_name)`.

**Usage in queries:**
- The composite index on `(dept_name, salary)` is very efficient for:
  ```sql
  SELECT * FROM employee WHERE dept_name = 'Finance' AND salary = 80000;
  ```
- It is less efficient for:
  ```sql
  SELECT * FROM employee WHERE salary = 80000;  -- cannot use index effectively
  ```

**Key rule:** For a composite index `(A, B)`, the index is most effective when the query filters on `A` first (equality), then on `B`.

### 45.4. Function-Based Indices

If queries use a function on a column (e.g., `UPPER(ename)`), a regular index on `ename` won't help.

**Solution:** Create a function-based index:
```sql
CREATE INDEX emp_upper_name_idx ON employee (UPPER(ename));
```

This index speeds up queries that use `UPPER(ename)` in the WHERE clause.

### 45.5. Index Privileges

To create an index, you need:
- **INDEX privilege** on the table (if you own the schema).
- **CREATE ANY INDEX privilege** (if you want to create an index on a table owned by someone else).
- **QUERY REWRITE privilege** (for function-based indices).
- **Quota for the TABLESPACE** where the index is stored.

### 45.6. Guidelines for Index Design (7 Ground Rules)

#### Rule 0: Indexes Lead to Access-Update Trade-off

- **More indexes** → faster queries, slower updates, more space.
- **Fewer indexes** → slower queries, faster updates, less space.
- Trade-off must be balanced based on the application's read/write ratio.

#### Rule 1: Index the Correct Tables

- Create an index if you frequently retrieve **less than 15%** of rows from a large table.
  - The threshold varies; it depends on table size, clustering, and query patterns.
- Index **foreign key** columns to speed up joins.
- Small tables may not need indexes (sequential scan is fine).
- **Primary and unique keys are automatically indexed** by the DBMS.

#### Rule 2: Index the Correct Columns

- **Regular (B-tree) index**: Good for columns with many distinct values (high cardinality).
  - Example: employee ID, social security number.
- **Bitmap index**: Good for columns with few distinct values (low cardinality).
  - Example: gender, semester, income level.
- **NULL values**: Use a special comparison to leverage the index.
  ```sql
  SELECT * FROM t WHERE amount > -9.99e125;
  ```
  (This uses the index and includes non-NULL values effectively.)

#### Rule 3: Limit the Number of Indexes per Table

- Each additional index adds overhead on every insert, delete, and update.
- A table that is **heavily read** can have more indexes.
- A table that is **heavily updated** should have fewer indexes.
- Remove unnecessary indexes.

#### Rule 4: Choose Column Order in Composite Index

- Put the column with **higher cardinality** (more distinct values) first.
- Put the column that is used in **equality conditions** first.
- Example: `(part_number, vendor_id)` is better than `(vendor_id, part_number)` if part_number has 1000 distinct values and vendor_id has only 5.

#### Rule 5: Gather Statistics

- Use `COMPUTE STATISTICS` to collect data about the index.
- Periodically refresh statistics (data distribution changes over time).
- Statistics help the query optimizer make informed decisions.

#### Rule 6: Drop Unused Indexes

- If an index is no longer needed (query patterns changed), drop it.
- Dropping a table automatically drops all its indexes.
- Use `DROP INDEX` to remove an index.

### 45.7. Summary of Index Design Process

1. **Start with the schema design** (normalization, ER model) — done once.
2. **Analyze query patterns**:
   - What queries are frequent?
   - What are the equality conditions?
   - What are the range conditions?
   - What are the join conditions?
3. **Choose the right index types**:
   - B-tree for high-cardinality columns.
   - Bitmap for low-cardinality columns.
   - Hash for point lookups.
4. **Create the necessary indexes**.
5. **Monitor performance**:
   - Use database statistics.
   - Identify slow queries.
6. **Adjust indexes**:
   - Add new indexes for frequent slow queries.
   - Drop unused or rarely used indexes.
7. **Repeat** as the data and query patterns evolve.

---

## Conclusion

Week 9 has provided a comprehensive understanding of how databases achieve fast data access through **indexing and hashing**:

- **Ordered indices** provide sorted access for range queries.
- **B+ trees** are the standard index structure, offering O(log n) search with self-balancing.
- **Hashing** provides O(1) average access for point queries but is poor for range queries.
- **Bitmap indices** are efficient for low-cardinality columns.
- **Index design** involves careful trade-offs between query performance, update cost, and space.

This knowledge is essential for database administrators and application developers to design efficient, scalable database systems. In the next weeks, we will explore query processing and optimization, where these index structures are used to execute SQL queries efficiently.